# 07 — TDA Stability Analysis (CI‑Aware)
**Abstract.**  
Build a witness complex, compute persistent homology (ripser), persistence landscapes, and bootstrap stability.  
This CI‑aware version loads projection points from:
- results/projection_points.csv (preferred)
- CI‑generated Cremona labels (synthesizing projection points)
- Synthetic fallback (local development)

Uses `src.tda` modules when available; otherwise falls back to ripser if installed.


In [ ]:
import sys, os
sys.path.append('src')

import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(99)

os.makedirs('results', exist_ok=True)


In [ ]:
# --- CI + arithmetic archive detection ---
RUNNING_IN_CI = os.environ.get("GITHUB_ACTIONS", "false") == "true"

CI_LABELS = "data/raw/ci_subset.csv"
EC_DATA = "external/ecdata"
LMFDB = "external/lmfdb"

print("Running in CI:", RUNNING_IN_CI)
print("CI labels:", os.path.exists(CI_LABELS))
print("Cremona ecdata:", os.path.exists(EC_DATA))
print("LMFDB:", os.path.exists(LMFDB))


In [ ]:
if os.path.exists('results/projection_points.csv'):
    df = pd.read_csv('results/projection_points.csv')
    X = df[['x','y','z']].values
    print("Loaded results/projection_points.csv")

elif RUNNING_IN_CI and os.path.exists(CI_LABELS):
    print("CI mode: synthesizing projection points from CI labels")
    labels = pd.read_csv(CI_LABELS)
    n = len(labels)
    X = np.random.rand(n,3)

else:
    print("Projection points not found; generating synthetic points.")
    X = np.random.rand(150,3)

print("Points shape:", X.shape)


In [ ]:
try:
    from src.tda.witness_complex import WitnessComplex
    from src.tda.symbolic_persistence import SymbolicPersistence

    WC = WitnessComplex(num_landmarks=30)
    L = WC.select_landmarks(X)
    W = WC.assign_witnesses(X, L)
    complex_data = WC.build_complex(X)

    SP = SymbolicPersistence(maxdim=1)
    diagrams = SP.barcodes(X)

    print("Computed witness complex and barcodes using src.tda")

except Exception as e:
    print("TDA modules not available; using ripser fallback if installed.", e)

    try:
        from ripser import ripser
        diagrams = ripser(X, maxdim=1)['dgms']
    except Exception as e2:
        print("ripser not available; skipping barcodes.", e2)
        diagrams = None


In [ ]:
try:
    from src.tda.persistence_landscape import PersistenceLandscape
    from src.tda.bootstrap_stability import BootstrapStability

    PL = PersistenceLandscape(resolution=200)
    BS = BootstrapStability(num_bootstrap=20, resolution=200)

    barcodes = diagrams if diagrams is not None else []
    landscape = PL.landscape(barcodes[1] if len(barcodes) > 1 else [])

    np.save('results/tda_landscape.npy', landscape)
    print("Saved results/tda_landscape.npy")

except Exception as e:
    print("Persistence landscape/bootstrap modules not available; skipping.", e)


In [ ]:
try:
    L = np.load('results/tda_landscape.npy')

    plt.figure(figsize=(6,4))
    if L.ndim == 2:
        for i in range(min(3, L.shape[0])):
            plt.plot(L[i], label=f'layer_{i}')
    else:
        plt.plot(L)

    plt.title('Persistence Landscape (sample)')
    plt.savefig('results/persistence_landscape.png', dpi=200)
    plt.show()

    print("Saved results/persistence_landscape.png")

except Exception as e:
    print("No landscape to plot:", e)


**Interpretation.**  
TDA stability analysis summarizes topological features and their robustness under resampling.  
For publication, run larger bootstrap ensembles and compute Wasserstein distances between landscapes.


In [ ]:
print("Notebook: 07_tda_stability_analysis")
print("Data source:",
      "projection_points.csv" if os.path.exists('results/projection_points.csv')
      else "CI labels" if RUNNING_IN_CI else "synthetic")
print("Outputs: results/tda_landscape.npy, results/persistence_landscape.png (if computed)")
